In [20]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time 
import yfinance as yf

In [56]:
tickers = pd.read_csv("../data/processed/quality_etfs.csv")['ticker']
# tickers = ['VOO']
# for ticker in tickers:
#     df = pd.read_csv(f"../data/raw/us market/prices/{ticker}.csv")
#     df = df[1:][['Date', 'Close', 'High', 'Low', 'Open', 'Volume']]  # Skip the first row
#     df['Date'] = pd.to_datetime(df['Date'])
    
#     df.to_csv(f"../data/official/us/price/{ticker}.csv", index=False)
#     # print(latest_date)

In [40]:
tickers = ['VOO']
for ticker in tickers:
    print(f"Updating data for {ticker}...")
    df = pd.read_csv(f"../data/official/us/price/{ticker}.csv")
    df = df[1:][['Date', 'Close', 'High', 'Low', 'Open', 'Volume']]  # Skip the first row
    print(df.tail())
    df['Date'] = pd.to_datetime(df['Date'])
    latest_date = df['Date'].max()
    start_date = latest_date + timedelta(days=1)
    end_date = datetime.now().date()

    new_data = yf.download(ticker, start=start_date, end=end_date)
    new_data.columns = new_data.columns.droplevel(0)
    # new_data.columns = ["Close", "High", "Low", "Open", "Volume"]
    # new_data = new_data.reset_index()
    print(new_data.tail()) # add those new data into the existing dataframe
    if not new_data.empty:
        updated_df = pd.concat([df, new_data], ignore_index= False)
        updated_df.to_csv(f"../data/official/us/price/{ticker}.csv", index=False)
    else:
        print(f"No new data for {ticker} from {start_date} to {end_date}")
    time.sleep(2)  # To avoid hitting rate limits

/var/folders/km/_b5m00hj0l91hg5vzm9rf92m0000gn/T/ipykernel_2885/2266089201.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  new_data = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['VOO']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-12-05 00:00:00 -> 2025-12-05)')


Updating data for VOO...
            Date       Close        High         Low        Open   Volume
3828  2025-11-28  628.409973  628.630005  625.710022  626.030029  2953200
3829  2025-12-01  625.520020  628.010010  624.099976  624.169983  9191600
3830  2025-12-02  626.580017  628.770020  624.619995  627.010010  5280600
3831  2025-12-03  628.770020  629.780029  624.950012  625.770020  6338400
3832  2025-12-04  629.299988  630.179993  626.479980  630.130005  5140500
Empty DataFrame
Columns: [(Adj Close, VOO), (Close, VOO), (High, VOO), (Low, VOO), (Open, VOO), (Volume, VOO)]
Index: []
No new data for VOO from 2025-12-05 00:00:00 to 2025-12-05


In [45]:
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import time

# tickers = ['VOO']

def clean_duplicates(df):
    """Remove duplicate rows using Date column."""
    df = df.drop_duplicates(subset=['Date'])
    df = df.sort_values('Date')
    return df

total = len(tickers)

from tqdm import tqdm

for ticker in tqdm(tickers, desc="Updating tickers"):

    # --- Load existing data ---
    df = pd.read_csv(f"../data/official/us/price/{ticker}.csv")
    df = df[1:][['Date', 'Close', 'High', 'Low', 'Open', 'Volume']]
    df['Date'] = pd.to_datetime(df['Date'])
    df = clean_duplicates(df)

    latest_date = df['Date'].max().date()
    end_date = datetime.now().date()

    # --- Skip request if up-to-date ---
    if latest_date >= end_date:
        print(f"No new dates for {ticker}. Latest date = {latest_date}. Skipping request.")
        continue

    start_date = latest_date + timedelta(days=1)
    print(f"Downloading new data from {start_date} to {end_date}...")

    try:
        new_data = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False
        )
    except Exception as e:
        print(f"Download error for {ticker}: {e}")
        continue

    # --- If Yahoo returns nothing ---
    if new_data.empty:
        print(f"No new data for {ticker}.")
        continue

    # --- Extract only the 5 columns you want ---
    required_cols = ["Close", "High", "Low", "Open", "Volume"]

    # Ensure all required columns exist
    new_data = new_data[required_cols].reset_index()

    # --- Combine + deduplicate ---
    updated_df = pd.concat([df, new_data], ignore_index=True)
    updated_df = clean_duplicates(updated_df)

    # --- Save back ---
    updated_df.to_csv(f"../data/official/us/price/{ticker}.csv", index=False)

    print(f"Updated {ticker}: +{len(new_data)} new rows.")

    time.sleep(2)   # keep your rate-limit protection


Updating tickers:   0%|          | 0/2202 [00:00<?, ?it/s]

Updated EWM: +1 new rows.


Updating tickers:   0%|          | 1/2202 [00:02<1:31:29,  2.49s/it]

Updated XLU: +1 new rows.


Updating tickers:   0%|          | 2/2202 [00:04<1:29:20,  2.44s/it]

...

Updating tickers: 100%|█████████▉| 2200/2202 [6:08:07<00:03,  1.97s/it]

Updated FRNW: +1 new rows.


Updating tickers: 100%|█████████▉| 2201/2202 [6:08:09<00:02,  2.06s/it]


1 Failed download:
['MOTE']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-09-27 -> 2025-12-06)')
Updating tickers: 100%|██████████| 2202/2202 [6:08:09<00:00, 10.03s/it]

No new data for MOTE.


In [57]:
import yfinance as yf
from datetime import datetime

def get_etf_info(ticker):
    """
    Fetch comprehensive ETF information from yfinance.
    Returns a dictionary with all available metadata.
    """
    try:
        etf = yf.Ticker(ticker)
        info = etf.info
        
        # Get top holdings (separate API call)
        try:
            holdings = etf.get_holdings()
            top_holdings = holdings.head(10).to_dict('records') if holdings is not None else None
        except:
            top_holdings = None
        
        # Convert Unix timestamp to readable date
        inception_date = None
        if info.get("fundInceptionDate"):
            try:
                inception_date = datetime.fromtimestamp(info.get("fundInceptionDate")).strftime('%Y-%m-%d')
            except:
                inception_date = info.get("fundInceptionDate")
        
        return {
            # Core Identifiers
            "ticker": info.get("symbol", ticker),
            "full_name": info.get("longName"),
            "short_name": info.get("shortName"),
            
            # Issuer & Classification
            "issuer": info.get("fundFamily"),
            "category": info.get("category"),
            "asset_class": info.get("quoteType"),  # Usually "ETF"
            "legal_type": info.get("legalType"),
            
            # Geographic & Market Info
            "region": info.get("region"),
            "market": info.get("market"),
            "exchange": info.get("exchange"),
            "currency": info.get("currency"),
            
            # Key Financial Metrics
            "expense_ratio": info.get("netExpenseRatio"),  # Use netExpenseRatio (more reliable)
            "aum": info.get("totalAssets") or info.get("netAssets"),
            "inception_date": inception_date,
            
            # Returns & Performance
            "dividend_yield": info.get("yield") or info.get("dividendYield"),
            "trailing_annual_dividend_rate": info.get("trailingAnnualDividendRate"),
            "ytd_return": info.get("ytdReturn"),
            "three_year_avg_return": info.get("threeYearAverageReturn"),
            "five_year_avg_return": info.get("fiveYearAverageReturn"),
            "trailing_three_month_returns": info.get("trailingThreeMonthReturns"),
            
            # Risk Metrics
            "beta_3year": info.get("beta3Year"),
            "trailing_pe": info.get("trailingPE"),
            "price_to_book": info.get("priceToBook"),
            
            # Price Data
            "nav_price": info.get("navPrice"),
            "regular_market_price": info.get("regularMarketPrice"),
            "fifty_two_week_high": info.get("fiftyTwoWeekHigh"),
            "fifty_two_week_low": info.get("fiftyTwoWeekLow"),
            "fifty_day_average": info.get("fiftyDayAverage"),
            "two_hundred_day_average": info.get("twoHundredDayAverage"),
            
            # Volume & Trading
            "average_volume": info.get("averageVolume"),
            "average_volume_10days": info.get("averageVolume10days"),
            
            # Description & Summary
            "description": info.get("longBusinessSummary"),
            
            # Holdings
            "top_holdings": top_holdings,
            
            # Metadata
            "source": "yfinance",
            "last_updated": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
    
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return {
            "ticker": ticker,
            "error": str(e),
            "source": "yfinance"
        }

# Example usage - Test with multiple tickers
# test_tickers = ["VOO", "QQQ", "AGG"]
etf_data = [get_etf_info(t) for t in tickers]

# Display as DataFrame for better visualization
import pandas as pd
df = pd.DataFrame(etf_data)
df

,ticker,full_name,short_name,issuer,category,asset_class,legal_type,region,market,exchange,...,fifty_two_week_high,fifty_two_week_low,fifty_day_average,two_hundred_day_average,average_volume,average_volume_10days,description,top_holdings,source,last_updated
0,EWM,iShares MSCI Malaysia ETF,iShares MSCI Malaysia Index Fun,iShares,Miscellaneous Region,ETF,Exchange Traded Fund,US,us_market,PCX,...,26.810,20.8000,26.108000,24.599750,259692,347240,The fund generally will invest at least 80% of...,None,yfinance,2025-12-06 22:14:12
1,XLU,The Utilities Select Sector SPDR Fund,SPDR Select Sector Fund - Utili,SPDR State Street Investment Management,Utilities,ETF,Exchange Traded Fund,US,us_market,PCX,...,46.885,35.5100,44.730800,41.704650,21228898,19346010,In seeking to track the performance of the ind...,None,yfinance,2025-12-06 22:14:12
2,EWL,iShares MSCI Switzerland ETF,iShares MSCI Switzerland ETF,iShares,Miscellaneous Region,ETF,Exchange Traded Fund,US,us_market,PCX,...,58.350,45.5600,56.605600,54.503550,407329,464500,The fund generally will invest at least 80% of...,None,yfinance,2025-12-06 22:14:12
3,EWQ,iShares MSCI France ETF,iShares MSCI France Index Fund,iShares,Miscellaneous Region,ETF,Exchange Traded Fund,US,us_market,PCX,...,45.460,35.2200,44.335600,42.523950,303104,280620,The fund generally will invest at least 80% of...,None,yfinance,2025-12-06 22:14:13
4,EWI,iShares MSCI Italy ETF,iShares MSCI Italy ETF,iShares,Miscellaneous Region,ETF,Exchange Traded Fund,US,us_market,PCX,...,54.530,35.2000,51.963400,47.961800,439362,388130,The fund generally will invest at least 80% of...,None,yfinance,2025-12-06 22:14:13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2197,FDHT,Fidelity Digital Health ETF,Fidelity Digital Health ETF,Fidelity Investments,Health,ETF,Exchange Traded Fund,US,us_market,BTS,...,22.400,15.7800,21.276012,20.605532,2764,3925,The fund normally invests at least 80% of asse...,None,yfinance,2025-12-06 22:26:24
2198,FCLD,Fidelity Cloud Computing ETF,Fidelity Cloud Computing ETF,Fidelity Investments,Technology,ETF,Exchange Traded Fund,US,us_market,BTS,...,31.930,20.0000,29.894880,27.548716,11256,8770,The fund normally invests at least 80% of its ...,None,yfinance,2025-12-06 22:26:24
2199,FDRV,Fidelity Electric Vehicles and Future Transpor...,Fidelity Electric Vehicles and,Fidelity Investments,Global Large-Stock Growth,ETF,Exchange Traded Fund,US,us_market,BTS,...,18.090,10.4100,17.067620,15.201005,8393,7100,The fund normally invests at least 80% of asse...,None,yfinance,2025-12-06 22:26:24
2200,FRNW,Fidelity Clean Energy ETF,Fidelity Clean Energy ETF,Fidelity Investments,Miscellaneous Sector,ETF,Exchange Traded Fund,US,us_market,BTS,...,22.340,11.2500,20.784100,16.688354,22967,17450,The fund normally invests at least 80% of asse...,None,yfinance,2025-12-06 22:26:25


In [58]:
df.to_csv("../data/official/us/etf_metadata.csv", index=False)